# Notebook 02: Interview Prompt Engineering — OFPHIL MSW Interview Assistant

**Project:** `ofphil-classification-v2`  
**Phase:** 05 — Model Selection and Training (`06_llm_interview_assistant`)  
**Objective:** Engineer system prompts, domain interview instructions, and contextual follow-up generators with strict clinical boundaries.

---

## 1. System Prompt & Guardrails Architecture

The MSW Interview Assistant must abide by non-negotiable ethical and clinical rules:

1. **Never Make Subsidy Decisions**: The LLM must never approve, reject, or classify charity eligibility. It does not calculate discount percentages or financial risk scores.
2. **Respect the Social Worker's Authority**: The assistant assists only by suggesting questions and organizing facts. The human Medical Social Worker retains full assessment responsibility.
3. **Zero Fabrication (No Hallucinations)**: Never assume unstated income, family members, or bills. Missing items must be recorded as unstated/null.
4. **Bilingual / Taglish Support**: Patients commonly communicate in a blend of English and Tagalog (Taglish). The model must understand conversational phrases (e.g., *"namamasada ng traysikel"*, *"wala po akong regular na sahod"*, *"ambagan lang po ang magkakapatid"*).

In [ ]:
# Core System Prompt and Governance Directives
import json
from pathlib import Path

MSW_SYSTEM_PROMPT = """You are the OFPHIL Medical Social Work (MSW) Interview Copilot, a specialized clinical AI assistant designed to support licensed social workers during patient intake interviews for charitable eye care and surgery.

YOUR PRIMARY ROLES:
1. Suggest clear, respectful, and culturally appropriate interview questions in English or Filipino/Taglish.
2. Identify ambiguities, vague numbers, or unclarified financial disclosures in patient responses.
3. Formulate gentle, context-aware follow-up questions to help the social worker clarify household circumstances.
4. Maintain a neutral, compassionate, and trauma-informed tone.

STRICT SAFETY & CLINICAL GOVERNANCE RULES:
- NEVER approve, reject, recommend, or assign a charity eligibility tier or subsidy percentage.
- NEVER replace or override the professional assessment of the Medical Social Worker.
- NEVER invent, assume, or extrapolate patient details (e.g., if a patient says 'I have no fixed job', do NOT assume income is zero; mark it as unclarified/informal).
- NEVER modify or sanitize patient quotes to make them look more or less favorable.
- NEVER calculate financial risk scores or eligibility formulas inside this model.
- ALWAYS distinguish between what the patient explicitly said versus what requires caseworker verification.
"""

print("System prompt defined. Character count:", len(MSW_SYSTEM_PROMPT))

## 2. Multi-Domain Structured Interview Prompts

We establish structured interview templates across four core intake domains:
1. **Household Structure & Dependents**
2. **Income & Informal Livelihood Disclosures**
3. **Living Conditions & Household Obligations (Rent/Utilities)**
4. **Clinical Documentation & Pre-Existing Assistance**

In [ ]:
# Domain interview templates dictionary
interview_domains = {
    "domain_1_household_and_dependents": {
        "topic": "Household Composition and Dependency",
        "starter_questions_en": [
            "How many family members currently live in your household?",
            "Are there young children, students, or elderly relatives who depend on your support?"
        ],
        "starter_questions_fil": [
            "Ilan po kayong magkakasama sa bahay sa kasalukuyan?",
            "Mayroon po bang mga anak, estudyante, o matatanda na umaasa sa inyo para sa pang-araw-araw?"
        ]
    },
    "domain_2_livelihood_and_income": {
        "topic": "Employment, Livelihood, and Monthly Earnings",
        "starter_questions_en": [
            "What is your current source of livelihood or occupation?",
            "On average, approximately how much does the household earn together in a typical month?"
        ],
        "starter_questions_fil": [
            "Ano po ang inyong kasalukuyang hanapbuhay o pinagkakakitaan?",
            "Sa karaniwang buwan, magkano po humigit-kumulang ang kabuuang pumapasok na kita sa inyong pamilya?"
        ]
    },
    "domain_3_expenses_and_housing": {
        "topic": "Housing Tenure and Monthly Obligations",
        "starter_questions_en": [
            "What is your current living arrangement (owned, renting, informal settler, or sharing)?",
            "Do you have regular monthly expenses for rent, electricity, or water?"
        ],
        "starter_questions_fil": [
            "Ano po ang estado ng inyong tinitirhan (sarili, umuupa, nakikitira, o informal settler)?",
            "Mayroon po ba kayong buwanang binabayaran para sa upa, kuryente, o tubig?"
        ]
    },
    "domain_4_medical_and_assistance": {
        "topic": "Clinical Procedure and External Assistance",
        "starter_questions_en": [
            "What specific eye surgery or procedure was recommended by your attending ophthalmologist?",
            "Have you received assistance or guarantees from PhilHealth, Malasakit, or your LGU for this procedure?"
        ],
        "starter_questions_fil": [
            "Anong operasyon o gamutan po sa mata ang inirekomenda ng inyong doktor?",
            "Nakahingi na po ba kayo ng tulong o guarantee letter mula sa PhilHealth, Malasakit Center, o LGU?"
        ]
    }
}

print(f"Configured {len(interview_domains)} intake domains with bilingual starters.")

## 3. Contextual Follow-up Generation Templates

Patients frequently respond with approximate or qualitative answers. The prompt below instructs the model to generate polite follow-up questions to resolve ambiguities.

In [ ]:
# Contextual follow-up generation prompt template
FOLLOWUP_PROMPT_TEMPLATE = """You are the OFPHIL MSW Interview Copilot.
Review the following snippet from a patient intake interview:

INTERVIEW CONTEXT:
- Domain: {domain_name}
- Last Caseworker Question: "{last_question}"
- Patient Response: "{patient_response}"

TASKS:
1. Detect any unclarified numbers, ambiguous timeframes, or missing details.
2. Formulate 2 polite, culturally respectful follow-up questions in the patient's preferred language ({preferred_language}).
3. Keep questions conversational, non-threatening, and concise.
4. Return strictly valid JSON conforming to the following structure:
{
  "detected_ambiguities": ["string"],
  "suggested_followups": [
    {
      "focus": "string",
      "question_text": "string"
    }
  ]
}
"""

print("Follow-up prompt template configured.")

## 4. Synthetic Validation Case Demonstration

We test the prompt template structure using a synthetic, anonymized patient utterance.

In [ ]:
# Synthetic demonstration: Building the prompt payload for an informal worker
sample_input = {
    "domain_name": "Livelihood and Monthly Earnings",
    "last_question": "Ano po ang inyong kasalukuyang hanapbuhay at magkano po ang karaniwang kita?",
    "patient_response": "Namasada po ako ng traysikel dati ma'am, pero ngayon dahil malabo na ang kanang mata ko, minsan na lang. Mga 200 o 300 pesos lang pag nakakabiyahe, pambili lang ng bigas.",
    "preferred_language": "Filipino / Taglish"
}

formatted_prompt = FOLLOWUP_PROMPT_TEMPLATE.format(**sample_input)
print("=== FORMATTED PROMPT PAYLOAD ===\n")
print(formatted_prompt[:450] + "\n...[truncated for display]...")